In [ ]:
# ═══════════════════════════════════════════════════════════
# BIM437 PIPELINE v2 — Profesyonel Akış
# LTD → preprocess (CRAE:128, CNN:112) → CR-AE → CNN → GradCAM
# Her çalıştırmada 1 klip, sonuç Drive'a kaydedilir
# ═══════════════════════════════════════════════════════════
import torch, torch.nn as nn, numpy as np, cv2
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from torchvision.models.video import r3d_18
from torch.amp import autocast
from pathlib import Path
import random, time

# ──────────────────────────────────────────────────────────
# AYARLAR
# ──────────────────────────────────────────────────────────
LTD_DIR        = Path("/content/drive/MyDrive/archive/LTD Dataset/LTD Dataset/Video Clips")
CNN_PATH       = "/content/drive/MyDrive/archive/Data_Annotated_Subset_Object_Detectors-CNN/training_final/cnn_multilabel_final_model/best_model.pth"
CRAE_PATH = "/content/drive/MyDrive/archive/Data_Subset_Autoencoders_Anomaly_Detectors-CRAE/models/crae_winter_finetuned2.pth"
RESULT_DIR     = Path("/content/drive/MyDrive/archive/pipeline_results")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

CLASSES        = ["normal","trespassing","loitering","object_abandonment"]
DEVICE         = torch.device("cuda")
N_FRAMES       = 16
CNN_SIZE       = 112
CRAE_SIZE      = 128
CRAE_THRESHOLD = 0.015

CNN_MEAN  = np.array([0.485,0.456,0.406], dtype=np.float32)
CNN_STD   = np.array([0.229,0.224,0.225], dtype=np.float32)
CRAE_MEAN = np.array([0.5,0.5,0.5],       dtype=np.float32)
CRAE_STD  = np.array([0.5,0.5,0.5],       dtype=np.float32)

COLORS = {
    "normal":             "#4CAF50",
    "trespassing":        "#F44336",
    "loitering":          "#FF9800",
    "object_abandonment": "#9C27B0",
}
ALERT_CLASSES = {"trespassing","object_abandonment"}

BG     = "#0D1117"
PANEL  = "#161B22"
BORDER = "#21262D"
TEXT   = "#C9D1D9"
MUTED  = "#8B949E"

# ──────────────────────────────────────────────────────────
# MODEL YÜKLEME
# ──────────────────────────────────────────────────────────
cnn = r3d_18(weights=None)
cnn.fc = nn.Sequential(nn.Dropout(0.5),
                        nn.Linear(cnn.fc.in_features,4))
ckpt = torch.load(CNN_PATH, map_location=DEVICE)
cnn.load_state_dict(ckpt["model_state"])
cnn  = cnn.to(DEVICE).eval()
print(f"✓ CNN  val_f1={ckpt['val_f1']:.4f}")

crae = None
try:
    crae_ckpt = torch.load(CRAE_PATH, map_location=DEVICE)
    # from your_module import CRAE
    # crae = CRAE(...); crae.load_state_dict(crae_ckpt["model_state"])
    # crae = crae.to(DEVICE).eval()
    print("✓ CR-AE yüklendi")
except Exception as e:
    print(f"⚠  CR-AE yok — her klip CNN'e gönderilir")

_act = {}
cnn.layer3[1].conv2.register_forward_hook(
    lambda m,i,o: _act.update({"v":o})
)
print("✓ Pipeline hazır\n")

# ──────────────────────────────────────────────────────────
# FONKSİYONLAR
# ──────────────────────────────────────────────────────────
def extract_frames(mp4_path, start):
    cap = cv2.VideoCapture(str(mp4_path))
    cap.set(cv2.CAP_PROP_POS_FRAMES, start)
    frames = []
    for _ in range(N_FRAMES):
        ret,f = cap.read()
        if not ret: break
        frames.append(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
    cap.release()
    return frames if len(frames)==N_FRAMES else None

def preprocess_cnn(frames):
    fm  = [cv2.resize(f,(CNN_SIZE,CNN_SIZE)) for f in frames]
    arr = (np.stack(fm).astype(np.float32)/255.0 - CNN_MEAN) / CNN_STD
    return torch.from_numpy(arr.transpose(3,0,1,2)).float()

def preprocess_crae(frames):
    fm  = [cv2.resize(f,(CRAE_SIZE,CRAE_SIZE)) for f in frames]
    arr = (np.stack(fm).astype(np.float32)/255.0 - CRAE_MEAN) / CRAE_STD
    return torch.from_numpy(arr.transpose(3,0,1,2)).float()

def infer_crae(t):
    if crae is None: return None, None
    with torch.no_grad():
        inp=t.unsqueeze(0).to(DEVICE); recon=crae(inp)
        err=torch.mean((inp-recon)**2).item()
    return err>CRAE_THRESHOLD, err

def infer_cnn(t):
    with torch.no_grad():
        with autocast('cuda'):
            logits = cnn(t.unsqueeze(0).to(DEVICE))
        probs = torch.sigmoid(logits).cpu().numpy()[0]
    pred    = int(probs.argmax())
    active  = [CLASSES[i] for i in range(4) if probs[i]>0.5]
    if not active: active=[CLASSES[pred]]
    alerts  = [c for c in active if c in ALERT_CLASSES]
    return pred, probs, active, alerts

def gradcam_pp(t, target):
    cnn.eval()
    inp=t.unsqueeze(0).to(DEVICE); logits=cnn(inp); act=_act["v"]
    try:
        g=torch.autograd.grad(logits[0,target],act,
                               retain_graph=True,create_graph=False)[0][0]
    except: return None
    a=act[0].detach(); gsq=g**2; gcu=g**3
    den=2*gsq+(a*gcu).sum(dim=(2,3),keepdim=True)
    den=torch.where(den!=0,den,torch.ones_like(den))
    w=(gsq/den*torch.relu(g)).mean(dim=(2,3),keepdim=True)
    cam=torch.relu((w*a).sum(0)).cpu().numpy()
    if cam.max()<=1e-6: return None
    return (cam-cam.min())/(cam.max()-cam.min()+1e-8)

def jet_overlay(frame, cam_2d, alpha=0.55):
    H,W=frame.shape[:2]
    cr=cv2.resize(cam_2d.astype(np.float32),(W,H))
    hm=cv2.cvtColor(cv2.applyColorMap((cr*255).astype(np.uint8),
                     cv2.COLORMAP_JET),cv2.COLOR_BGR2RGB)
    return (alpha*hm+(1-alpha)*frame).astype(np.uint8)

def cam_fi(cam,fi):
    T=cam.shape[0]; return cam[min(int(fi*T/N_FRAMES),T-1)]

# ──────────────────────────────────────────────────────────
# GÖRSEL — Profesyonel panel
# ──────────────────────────────────────────────────────────
def render_and_save(frames, probs, active, alerts, cnn_t,
                    crae_res, info, run_id):

    active_idx = [CLASSES.index(c) for c in active if c in CLASSES]
    n_cam_rows = len(active_idx)
    # Layout: başlık + 16 frame + gradcam satırları + alt bar
    fig = plt.figure(figsize=(24, 3.5 + n_cam_rows*3.2 + 2.8),
                     facecolor=BG)

    total_rows  = 1 + n_cam_rows + 1
    height_rats = [1.5] + [1.8]*n_cam_rows + [1.2]
    gs = gridspec.GridSpec(total_rows, 16,
                           figure=fig, hspace=0.06, wspace=0.03,
                           height_ratios=height_rats)

    # ── Satır 0: 16 frame ───────────────────────────────
    for fi in range(N_FRAMES):
        ax = fig.add_subplot(gs[0, fi])
        ax.imshow(frames[fi])
        ax.axis('off')
        ax.set_title(str(fi+1), fontsize=6, color=MUTED,
                     pad=1, fontfamily='monospace')
        for sp in ax.spines.values():
            sp.set_visible(True); sp.set_edgecolor(BORDER)
            sp.set_linewidth(0.4)

    # ── Satır 1..N: GradCAM (5 temsili frame) ───────────
    SHOW = [0,3,7,11,15]
    for row_i, ai in enumerate(active_idx):
        cname = CLASSES[ai]
        color = COLORS[cname]
        cam   = gradcam_pp(cnn_t, ai)

        for col_i, fi in enumerate(SHOW):
            span_s = col_i*3
            span_e = span_s+3 if col_i<4 else 16
            ax = fig.add_subplot(gs[1+row_i, span_s:span_e])
            frame = frames[fi]
            if cam is not None:
                ax.imshow(jet_overlay(frame, cam_fi(cam,fi)))
            else:
                ax.imshow(frame)
            ax.axis('off')
            if col_i==0:
                ax.set_ylabel(
                    cname.replace('_','\n').upper(),
                    fontsize=6.5, color=color,
                    fontfamily='monospace',
                    rotation=0, ha='right', va='center',
                    labelpad=58)
            for sp in ax.spines.values():
                sp.set_visible(True)
                sp.set_edgecolor(color if col_i==0 else BORDER)
                sp.set_linewidth(0.8 if col_i==0 else 0.3)

        # Prob etiketi sağ
        ax_p = fig.add_subplot(gs[1+row_i, 15])
        ax_p.set_facecolor(PANEL); ax_p.axis('off')
        ax_p.text(0.5,0.5, f"{probs[ai]:.2f}",
                  ha='center',va='center',
                  fontsize=13,fontweight='bold',color=color,
                  fontfamily='monospace',
                  transform=ax_p.transAxes)

    # ── Alt panel ────────────────────────────────────────
    # Prob bars (0:8)
    ax_b = fig.add_subplot(gs[-1,:8])
    ax_b.set_facecolor(PANEL)
    bcolors = [COLORS[c] for c in CLASSES]
    ax_b.barh(range(4), probs, color=bcolors, alpha=0.8,
              height=0.5, edgecolor='none')
    ax_b.set_yticks(range(4))
    ax_b.set_yticklabels([c.replace('_',' ').upper() for c in CLASSES],
                          fontsize=6.5, color=MUTED, fontfamily='monospace')
    ax_b.set_xlim(0,1)
    ax_b.axvline(0.5, color=BORDER, lw=0.8, ls='--')
    ax_b.tick_params(axis='x', colors=MUTED, labelsize=6)
    for sp in ['top','right','left']: ax_b.spines[sp].set_visible(False)
    ax_b.spines['bottom'].set_color(BORDER)
    ax_b.set_facecolor(PANEL)
    for i,p in enumerate(probs):
        ax_b.text(min(p+0.02,0.93),i,f"{p:.3f}",
                  va='center',fontsize=6.5,color=TEXT,
                  fontfamily='monospace')
    ax_b.set_title("CNN  Sigmoid", fontsize=7, color=MUTED,
                   fontfamily='monospace', pad=3)

    # CR-AE box (8:12)
    ax_cr = fig.add_subplot(gs[-1,8:12])
    ax_cr.set_facecolor(PANEL); ax_cr.axis('off')
    if crae_res[1] is None:
        cr_txt = "CR-AE\n—  (model yok)"; cr_col = MUTED
    else:
        cr_txt = (f"CR-AE\nerr={crae_res[1]:.5f}\n"
                  f"thr={CRAE_THRESHOLD}\n"
                  f"{'● ANOMALİ' if crae_res[0] else '○ NORMAL'}")
        cr_col = "#F44336" if crae_res[0] else "#4CAF50"
    ax_cr.text(0.5,0.5,cr_txt,ha='center',va='center',
               fontsize=7.5,color=cr_col,fontfamily='monospace',
               transform=ax_cr.transAxes,
               bbox=dict(boxstyle='round,pad=0.5',
                         facecolor=BG,edgecolor=cr_col,alpha=0.9))

    # Karar (12:16)
    ax_d = fig.add_subplot(gs[-1,12:])
    ax_d.set_facecolor(PANEL); ax_d.axis('off')
    if alerts:
        dt  = "🚨  ALERT\n" + "\n".join([a.replace('_',' ').upper() for a in alerts])
        dc  = "#FF4444"; db = "#2D1010"; de = "#FF4444"
    else:
        dt  = "✓  NORMAL\nbildirim yok"
        dc  = "#4CAF50"; db = "#101D10"; de = "#4CAF50"
    ax_d.text(0.5,0.5,dt,ha='center',va='center',
              fontsize=9,fontweight='bold',color=dc,
              fontfamily='monospace',transform=ax_d.transAxes,
              bbox=dict(boxstyle='round,pad=0.6',
                        facecolor=db,edgecolor=de,linewidth=1.5))

    # ── Üst başlık şeridi ───────────────────────────────
    active_str = " + ".join([c.replace('_',' ').upper() for c in active])
    fig.text(0.01, 0.995,
             f"BIM437 · DETECTION PIPELINE · Run #{run_id:04d}",
             fontsize=9, color=MUTED, fontfamily='monospace',
             va='top')
    fig.text(0.5, 0.995,
             f"{info['day']} / {info['video']}   "
             f"@ frame {info['start']}  ({info['time']})",
             fontsize=8, color=MUTED, fontfamily='monospace',
             va='top', ha='center')
    fig.text(0.99, 0.995,
             f"CNN → [{active_str}]",
             fontsize=8,
             color="#F44336" if alerts else "#4CAF50",
             fontfamily='monospace', va='top', ha='right')

    plt.tight_layout(rect=[0,0,1,0.993])

    status = "ALERT" if alerts else "NORMAL"
    ac_str = "+".join([a[:3].upper() for a in alerts]) if alerts else "NRM"
    fname  = (f"run{run_id:04d}_{status}_{ac_str}_"
              f"{info['day']}_{info['video'][:12]}_f{info['start']}.png")
    fpath  = RESULT_DIR/fname
    plt.savefig(str(fpath),dpi=130,bbox_inches='tight',facecolor=BG)
    plt.show()
    return str(fpath)

# ──────────────────────────────────────────────────────────
# ANA PIPELINE
# ──────────────────────────────────────────────────────────
def run_pipeline():
    SEP = "═"*60
    print(f"\n{SEP}")
    print("  BIM437  ·  THERMAL ANOMALY DETECTION  PIPELINE")
    print(f"{SEP}")

    # 1. LTD random klip
    day_dirs = [d for d in LTD_DIR.iterdir() if d.is_dir()]
    day_dir  = random.choice(day_dirs)
    mp4s     = list(day_dir.glob("*.mp4"))
    if not mp4s: print("  ✗ MP4 yok"); return
    mp4 = random.choice(mp4s)
    cap = cv2.VideoCapture(str(mp4))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps   = cap.get(cv2.CAP_PROP_FPS)
    cap.release()
    start  = random.randint(0, max(0,total-N_FRAMES-1))
    frames = extract_frames(mp4, start)
    if frames is None: print("  ✗ Frame yok"); return

    info = {"day":day_dir.name,"video":mp4.name,
            "start":start,"time":f"{start/fps:.1f}sn" if fps>0 else "?"}
    print(f"\n  📁 {info['day']}  /  {info['video']}")
    print(f"  🎞  frame={start}  t={info['time']}  ({total} toplam)")

    # 2. Preprocess
    print(f"\n  [1/4] Preprocess ──────────────────────────────")
    cnn_t  = preprocess_cnn(frames)
    crae_t = preprocess_crae(frames)
    print(f"  CNN  tensor : {tuple(cnn_t.shape)}  (normalized ImageNet)")
    print(f"  CRAE tensor : {tuple(crae_t.shape)}  (normalized ±1)")

    # 3. CR-AE
    print(f"\n  [2/4] CR-AE ────────────────────────────────────")
    t0 = time.time()
    anomali, recon_err = infer_crae(crae_t)
    elapsed = time.time()-t0
    if recon_err is not None:
        bar_len = int((recon_err/CRAE_THRESHOLD)*20)
        bar     = "█"*min(bar_len,30)
        print(f"  err={recon_err:.5f}  thr={CRAE_THRESHOLD}  {elapsed:.3f}sn")
        print(f"  [{bar}]  {'ANOMALİ ✓' if anomali else 'NORMAL ✗'}")
        if not anomali:
            print(f"\n  ╔══════════════════════════════════╗")
            print(f"  ║  CR-AE: NORMAL  →  klip atlandı  ║")
            print(f"  ╚══════════════════════════════════╝\n")
            return
    else:
        print(f"  CR-AE yok → devam ({elapsed:.3f}sn)")

    # 4. CNN
    print(f"\n  [3/4] CNN ──────────────────────────────────────")
    t0 = time.time()
    pred_idx, probs, active, alerts = infer_cnn(cnn_t)
    elapsed = time.time()-t0
    print(f"  {elapsed:.3f}sn")
    for i,c in enumerate(CLASSES):
        bar = "█"*int(probs[i]*30)
        star= " ◄ ALERT" if c in ALERT_CLASSES and probs[i]>0.5 else ""
        print(f"  {c:<22} [{bar:<30}] {probs[i]:.3f}{star}")

    # 5. Karar
    print(f"\n  [4/4] Karar ────────────────────────────────────")
    if alerts:
        print(f"  ┌─────────────────────────────────────┐")
        print(f"  │  🚨  ALERT: {str(alerts):<26}│")
        print(f"  │  Operatöre bildirim gönderiliyor    │")
        print(f"  └─────────────────────────────────────┘")
    else:
        print(f"  ┌─────────────────────────────────────┐")
        print(f"  │  ✓  NORMAL — operatör bildirimi yok │")
        print(f"  └─────────────────────────────────────┘")

    # 6. Render + Save
    run_id = len(list(RESULT_DIR.glob("run*.png")))+1
    fpath  = render_and_save(
        frames, probs, active, alerts, cnn_t,
        (anomali, recon_err), info, run_id
    )
    print(f"\n  ✓ Kaydedildi  →  {Path(fpath).name}")
    print(f"  📂 {RESULT_DIR}")
    print(f"  Toplam sonuç: {len(list(RESULT_DIR.glob('run*.png')))}")
    print(f"\n{SEP}\n")

# ──────────────────────────────────────────────────────────
run_pipeline()

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# BIM437  ·  FULL EVALUATION PIPELINE  v3
# Test setindeki TÜM klipler:
#   preprocess (CRAE:128, CNN:112) → CR-AE filtresi → CNN → GradCAM
#   → per-clip PNG  +  özet metrics raporu  (Drive'a kaydedilir)
# ═══════════════════════════════════════════════════════════════════
import torch, torch.nn as nn, numpy as np, cv2
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from torchvision.models.video import r3d_18
from torch.amp import autocast
from pathlib import Path
import json, time, random
from tqdm import tqdm
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report,
    roc_auc_score, ConfusionMatrixDisplay
)
import warnings
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────
# AYARLAR  —  yolları kendi Drive yapınıza göre düzenleyin
# ─────────────────────────────────────────────────────────────────
CNN_PATH   = "/content/drive/MyDrive/archive/Data_Annotated_Subset_Object_Detectors-CNN/training_final/cnn_multilabel_final_model/best_model.pth"
CRAE_PATH  = "/content/drive/MyDrive/archive/Data_Subset_Autoencoders_Anomaly_Detectors-CRAE/models/crae_winter_finetuned2.pth"

# Test seti: her alt klasör bir sınıf adı taşıyorsa → otomatik etiket
# Düz video listesi varsa LABEL_CSV kullanın
TEST_DIR   = Path("/content/drive/MyDrive/archive/test_set")          # video klasörü
LABEL_CSV  = None   # örn: "/content/drive/MyDrive/archive/test_labels.csv"

RESULT_DIR = Path("/content/drive/MyDrive/archive/eval_results_v3")
RESULT_DIR.mkdir(parents=True, exist_ok=True)
(RESULT_DIR / "clips").mkdir(exist_ok=True)

CLASSES        = ["normal", "trespassing", "loitering", "object_abandonment"]
ALERT_CLASSES  = {"trespassing", "object_abandonment"}
DEVICE         = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_FRAMES       = 16
CNN_SIZE       = 112
CRAE_SIZE      = 128
CRAE_THRESHOLD = 0.015
CNN_THRESHOLD  = 0.5

CNN_MEAN  = np.array([0.485, 0.456, 0.406], dtype=np.float32)
CNN_STD   = np.array([0.229, 0.224, 0.225], dtype=np.float32)
CRAE_MEAN = np.array([0.5,   0.5,   0.5  ], dtype=np.float32)
CRAE_STD  = np.array([0.5,   0.5,   0.5  ], dtype=np.float32)

COLORS = {
    "normal":             "#4CAF50",
    "trespassing":        "#F44336",
    "loitering":          "#FF9800",
    "object_abandonment": "#9C27B0",
}
BG = "#0D1117"; PANEL = "#161B22"; BORDER = "#21262D"
TEXT = "#C9D1D9"; MUTED = "#8B949E"

# ─────────────────────────────────────────────────────────────────
# MODEL YÜKLEME
# ─────────────────────────────────────────────────────────────────
print("  [INIT] Modeller yükleniyor …")

# — CNN (R3D-18) —
cnn = r3d_18(weights=None)
cnn.fc = nn.Sequential(nn.Dropout(0.5),
                        nn.Linear(cnn.fc.in_features, 4))
ckpt = torch.load(CNN_PATH, map_location=DEVICE)
cnn.load_state_dict(ckpt["model_state"])
cnn = cnn.to(DEVICE).eval()
print(f"  ✓ CNN  val_f1={ckpt.get('val_f1', '?')}")

# — GradCAM hook —
_act = {}
cnn.layer3[1].conv2.register_forward_hook(
    lambda m, i, o: _act.update({"v": o})
)

# — CR-AE —
# !! Kendi CRAE sınıfınızı buraya import edin
# from your_crae_module import CRAE
crae = None
try:
    crae_ckpt = torch.load(CRAE_PATH, map_location=DEVICE)
    # crae = CRAE(...)
    # crae.load_state_dict(crae_ckpt["model_state"])
    # crae = crae.to(DEVICE).eval()
    print("  ✓ CR-AE yüklendi")
except Exception as e:
    print(f"  ⚠  CR-AE yüklenemedi → her klip CNN'e gönderilir  ({e})")

print(f"  ✓ Device: {DEVICE}\n")

# ─────────────────────────────────────────────────────────────────
# YARDIMCI FONKSİYONLAR
# ─────────────────────────────────────────────────────────────────
def load_test_clips():
    """
    Test kliplerine ait (path, label_idx) listesi döndürür.
    Önce LABEL_CSV'e bakar; yoksa klasör adından etiket çıkarır.
    Klasör yapısı:  test_set/<class_name>/<clip>.mp4
    """
    clips = []
    if LABEL_CSV:
        import csv
        with open(LABEL_CSV) as f:
            for row in csv.DictReader(f):
                p = Path(row["path"])
                l = CLASSES.index(row["label"])
                if p.exists():
                    clips.append((p, l))
        print(f"  CSV'den {len(clips)} klip yüklendi.")
        return clips

    for cls_idx, cls_name in enumerate(CLASSES):
        cls_dir = TEST_DIR / cls_name
        if not cls_dir.exists():
            continue
        for mp4 in sorted(cls_dir.glob("*.mp4")):
            clips.append((mp4, cls_idx))
    if not clips:                          # düz dizin — etiket yok
        for mp4 in sorted(TEST_DIR.rglob("*.mp4")):
            clips.append((mp4, -1))        # bilinmeyen etiket
    print(f"  {len(clips)} test klibi bulundu.")
    return clips


def extract_frames(mp4_path, start=0):
    cap = cv2.VideoCapture(str(mp4_path))
    cap.set(cv2.CAP_PROP_POS_FRAMES, start)
    frames = []
    for _ in range(N_FRAMES):
        ret, f = cap.read()
        if not ret:
            break
        frames.append(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
    cap.release()
    if len(frames) < N_FRAMES:
        if len(frames) == 0:
            return None
        # eksik kareleri son kare ile doldur
        while len(frames) < N_FRAMES:
            frames.append(frames[-1].copy())
    return frames


def preprocess_cnn(frames):
    fm  = [cv2.resize(f, (CNN_SIZE, CNN_SIZE)) for f in frames]
    arr = (np.stack(fm).astype(np.float32) / 255.0 - CNN_MEAN) / CNN_STD
    return torch.from_numpy(arr.transpose(3, 0, 1, 2)).float()


def preprocess_crae(frames):
    fm  = [cv2.resize(f, (CRAE_SIZE, CRAE_SIZE)) for f in frames]
    arr = (np.stack(fm).astype(np.float32) / 255.0 - CRAE_MEAN) / CRAE_STD
    return torch.from_numpy(arr.transpose(3, 0, 1, 2)).float()


def infer_crae(t):
    """(is_anomaly: bool|None, recon_error: float|None)"""
    if crae is None:
        return None, None
    with torch.no_grad():
        inp   = t.unsqueeze(0).to(DEVICE)
        recon = crae(inp)
        err   = torch.mean((inp - recon) ** 2).item()
    return err > CRAE_THRESHOLD, err


def infer_cnn(t):
    """pred_idx, probs[4], active_classes, alert_classes"""
    with torch.no_grad():
        with autocast('cuda'):
            logits = cnn(t.unsqueeze(0).to(DEVICE))
        probs = torch.sigmoid(logits).cpu().numpy()[0]
    pred    = int(probs.argmax())
    active  = [CLASSES[i] for i in range(4) if probs[i] > CNN_THRESHOLD]
    if not active:
        active = [CLASSES[pred]]
    alerts  = [c for c in active if c in ALERT_CLASSES]
    return pred, probs, active, alerts


def gradcam_pp(t, target):
    """GradCAM++ — (T, H, W) float array ya da None"""
    cnn.eval()
    inp    = t.unsqueeze(0).to(DEVICE)
    logits = cnn(inp)
    act    = _act["v"]
    try:
        g = torch.autograd.grad(logits[0, target], act,
                                retain_graph=True,
                                create_graph=False)[0][0]
    except Exception:
        return None
    a   = act[0].detach()
    gsq = g ** 2
    gcu = g ** 3
    den = 2 * gsq + (a * gcu).sum(dim=(2, 3), keepdim=True)
    den = torch.where(den != 0, den, torch.ones_like(den))
    w   = (gsq / den * torch.relu(g)).mean(dim=(2, 3), keepdim=True)
    cam = torch.relu((w * a).sum(0)).cpu().numpy()
    if cam.max() <= 1e-6:
        return None
    return (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)


def jet_overlay(frame, cam_2d, alpha=0.55):
    H, W = frame.shape[:2]
    cr   = cv2.resize(cam_2d.astype(np.float32), (W, H))
    hm   = cv2.cvtColor(
        cv2.applyColorMap((cr * 255).astype(np.uint8), cv2.COLORMAP_JET),
        cv2.COLOR_BGR2RGB)
    return (alpha * hm + (1 - alpha) * frame).astype(np.uint8)


def cam_at_frame(cam, fi):
    T = cam.shape[0]
    return cam[min(int(fi * T / N_FRAMES), T - 1)]

# ─────────────────────────────────────────────────────────────────
# PER-CLIP PNG
# ─────────────────────────────────────────────────────────────────
SHOW_FRAMES = [0, 3, 7, 11, 15]

def render_clip(frames, probs, active, alerts,
                cnn_t, crae_is_anom, crae_err,
                gt_label, clip_name, clip_idx):
    active_idx  = [CLASSES.index(c) for c in active if c in CLASSES]
    n_cam_rows  = len(active_idx)
    fig = plt.figure(figsize=(24, 3.5 + n_cam_rows * 3.2 + 2.8),
                     facecolor=BG)
    total_rows  = 1 + n_cam_rows + 1
    height_rats = [1.5] + [1.8] * n_cam_rows + [1.2]
    gs = gridspec.GridSpec(total_rows, 16, figure=fig,
                           hspace=0.06, wspace=0.03,
                           height_ratios=height_rats)

    # ── Satır 0 : 16 ham kare ────────────────────────────
    for fi in range(N_FRAMES):
        ax = fig.add_subplot(gs[0, fi])
        ax.imshow(frames[fi]); ax.axis("off")
        ax.set_title(str(fi + 1), fontsize=6, color=MUTED, pad=1,
                     fontfamily="monospace")
        for sp in ax.spines.values():
            sp.set_visible(True); sp.set_edgecolor(BORDER)
            sp.set_linewidth(0.4)

    # ── Satır 1..N : GradCAM ─────────────────────────────
    for row_i, ai in enumerate(active_idx):
        cname = CLASSES[ai]; color = COLORS[cname]
        cam   = gradcam_pp(cnn_t, ai)
        for col_i, fi in enumerate(SHOW_FRAMES):
            span_s = col_i * 3
            span_e = span_s + 3 if col_i < 4 else 16
            ax = fig.add_subplot(gs[1 + row_i, span_s:span_e])
            if cam is not None:
                ax.imshow(jet_overlay(frames[fi], cam_at_frame(cam, fi)))
            else:
                ax.imshow(frames[fi])
            ax.axis("off")
            if col_i == 0:
                ax.set_ylabel(cname.replace("_", "\n").upper(),
                              fontsize=6.5, color=color,
                              fontfamily="monospace",
                              rotation=0, ha="right", va="center",
                              labelpad=58)
            for sp in ax.spines.values():
                sp.set_visible(True)
                sp.set_edgecolor(color if col_i == 0 else BORDER)
                sp.set_linewidth(0.8 if col_i == 0 else 0.3)
        # prob etiketi
        ax_p = fig.add_subplot(gs[1 + row_i, 15])
        ax_p.set_facecolor(PANEL); ax_p.axis("off")
        ax_p.text(0.5, 0.5, f"{probs[ai]:.2f}",
                  ha="center", va="center", fontsize=13,
                  fontweight="bold", color=color,
                  fontfamily="monospace",
                  transform=ax_p.transAxes)

    # ── Alt panel ─────────────────────────────────────────
    ax_b = fig.add_subplot(gs[-1, :8])
    ax_b.set_facecolor(PANEL)
    ax_b.barh(range(4), probs, color=[COLORS[c] for c in CLASSES],
              alpha=0.8, height=0.5, edgecolor="none")
    ax_b.set_yticks(range(4))
    ax_b.set_yticklabels([c.replace("_", " ").upper() for c in CLASSES],
                          fontsize=6.5, color=MUTED,
                          fontfamily="monospace")
    ax_b.set_xlim(0, 1)
    ax_b.axvline(0.5, color=BORDER, lw=0.8, ls="--")
    ax_b.tick_params(axis="x", colors=MUTED, labelsize=6)
    for sp in ["top", "right", "left"]:
        ax_b.spines[sp].set_visible(False)
    ax_b.spines["bottom"].set_color(BORDER)
    ax_b.set_facecolor(PANEL)
    for i, p in enumerate(probs):
        ax_b.text(min(p + 0.02, 0.93), i, f"{p:.3f}",
                  va="center", fontsize=6.5, color=TEXT,
                  fontfamily="monospace")
    ax_b.set_title("CNN Sigmoid", fontsize=7, color=MUTED,
                   fontfamily="monospace", pad=3)

    # CR-AE kutusu
    ax_cr = fig.add_subplot(gs[-1, 8:12])
    ax_cr.set_facecolor(PANEL); ax_cr.axis("off")
    if crae_err is None:
        cr_txt = "CR-AE\n— (model yok)"; cr_col = MUTED
    else:
        cr_txt = (f"CR-AE\nerr={crae_err:.5f}\n"
                  f"thr={CRAE_THRESHOLD}\n"
                  f"{'● ANOMALİ' if crae_is_anom else '○ NORMAL'}")
        cr_col = "#F44336" if crae_is_anom else "#4CAF50"
    ax_cr.text(0.5, 0.5, cr_txt, ha="center", va="center",
               fontsize=7.5, color=cr_col, fontfamily="monospace",
               transform=ax_cr.transAxes,
               bbox=dict(boxstyle="round,pad=0.5",
                         facecolor=BG, edgecolor=cr_col, alpha=0.9))

    # Karar + GT
    ax_d = fig.add_subplot(gs[-1, 12:])
    ax_d.set_facecolor(PANEL); ax_d.axis("off")
    gt_str = CLASSES[gt_label] if gt_label >= 0 else "bilinmiyor"
    pred_str = active[0] if active else "normal"
    correct  = (gt_label < 0) or (CLASSES[gt_label] in active)
    match_sym = "✓" if correct else "✗"
    match_col = "#4CAF50" if correct else "#F44336"
    if alerts:
        dt = f"🚨  ALERT\n{chr(10).join([a.replace('_',' ').upper() for a in alerts])}"
        dc = "#FF4444"; db = "#2D1010"; de = "#FF4444"
    else:
        dt = "✓  NORMAL\nbildirim yok"
        dc = "#4CAF50"; db = "#101D10"; de = "#4CAF50"
    ax_d.text(0.5, 0.65, dt, ha="center", va="center",
              fontsize=9, fontweight="bold", color=dc,
              fontfamily="monospace", transform=ax_d.transAxes,
              bbox=dict(boxstyle="round,pad=0.5",
                        facecolor=db, edgecolor=de, linewidth=1.5))
    ax_d.text(0.5, 0.18, f"GT: {gt_str}   {match_sym}",
              ha="center", va="center",
              fontsize=8, color=match_col,
              fontfamily="monospace", transform=ax_d.transAxes)

    # Başlık şeridi
    fig.text(0.01, 0.997,
             f"BIM437 · EVAL #{clip_idx:04d}",
             fontsize=9, color=MUTED, fontfamily="monospace", va="top")
    fig.text(0.5, 0.997,
             clip_name,
             fontsize=8, color=MUTED, fontfamily="monospace",
             va="top", ha="center")
    fig.text(0.99, 0.997,
             f"CNN → [{' + '.join(active)}]",
             fontsize=8,
             color="#F44336" if alerts else "#4CAF50",
             fontfamily="monospace", va="top", ha="right")

    plt.tight_layout(rect=[0, 0, 1, 0.994])
    status = "ALERT" if alerts else "NORMAL"
    fname  = (f"clip{clip_idx:04d}_{status}_"
              f"GT{CLASSES[gt_label][:4] if gt_label>=0 else 'UNK'}"
              f"_{clip_name[:20]}.png")
    fpath  = RESULT_DIR / "clips" / fname
    plt.savefig(str(fpath), dpi=110, bbox_inches="tight", facecolor=BG)
    plt.close(fig)
    return str(fpath)

# ─────────────────────────────────────────────────────────────────
# METRİK RAPORU
# ─────────────────────────────────────────────────────────────────
def compute_and_save_metrics(records):
    """
    records: list of dicts with keys
        gt_label, pred_label, probs[4],
        crae_is_anom, crae_err, skipped_by_crae, alerts
    """
    has_gt = any(r["gt_label"] >= 0 for r in records)
    total  = len(records)
    skipped = sum(1 for r in records if r["skipped_by_crae"])

    print(f"\n{'═'*62}")
    print("  BIM437  ·  FULL EVALUATION METRICS")
    print(f"{'═'*62}")
    print(f"  Toplam klip       : {total}")
    print(f"  CRAE filtresi ile atlandı: {skipped}  "
          f"({100*skipped/total:.1f}%)")
    print(f"  CNN'e gönderilen  : {total - skipped}")

    if not has_gt:
        print("\n  ⚠  Ground-truth etiket bulunamadı.")
        print("     Etiketli test seti için LABEL_CSV veya")
        print("     sınıf adlı alt klasörler kullanın.\n")
        return

    # --- sadece etiketli kayıtlar ---
    labeled = [r for r in records if r["gt_label"] >= 0]
    y_true  = np.array([r["gt_label"]  for r in labeled])
    y_pred  = np.array([r["pred_label"] for r in labeled])
    probs   = np.array([r["probs"]      for r in labeled])   # (N,4)

    # ── 1. Sınıf bazlı metrikler ─────────────────────────
    print("\n  ── Per-Class Metrics ──────────────────────────")
    report = classification_report(
        y_true, y_pred,
        target_names=CLASSES,
        digits=4, zero_division=0
    )
    print(report)

    # ── 2. Macro / Weighted ortalama ─────────────────────
    acc = accuracy_score(y_true, y_pred)
    p_mac, r_mac, f1_mac, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro",  zero_division=0)
    p_w,   r_w,   f1_w,   _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0)
    print(f"  ── Özet ───────────────────────────────────────")
    print(f"  Accuracy (micro)  : {acc:.4f}")
    print(f"  Macro  P/R/F1     : {p_mac:.4f} / {r_mac:.4f} / {f1_mac:.4f}")
    print(f"  Weighted P/R/F1   : {p_w:.4f}   / {r_w:.4f}   / {f1_w:.4f}")

    # ── 3. ROC-AUC (one-vs-rest) ─────────────────────────
    try:
        auc_scores = {}
        for ci, cname in enumerate(CLASSES):
            binary_gt = (y_true == ci).astype(int)
            if binary_gt.sum() > 0:
                auc_scores[cname] = roc_auc_score(binary_gt, probs[:, ci])
        print(f"\n  ── ROC-AUC (one-vs-rest) ───────────────────")
        for cname, auc in auc_scores.items():
            bar = "█" * int(auc * 30)
            print(f"  {cname:<22} [{bar:<30}] {auc:.4f}")
        macro_auc = np.mean(list(auc_scores.values()))
        print(f"  {'Macro AUC':<22}  {' '*31} {macro_auc:.4f}")
    except Exception as e:
        print(f"  ROC-AUC hesaplanamadı: {e}")

    # ── 4. Alert-level (binary) metrik ───────────────────
    # Alert = trespassing veya object_abandonment
    alert_idxs = {CLASSES.index(c) for c in ALERT_CLASSES}
    y_alert_true = np.isin(y_true, list(alert_idxs)).astype(int)
    y_alert_pred = np.isin(y_pred, list(alert_idxs)).astype(int)
    p_a, r_a, f1_a, _ = precision_recall_fscore_support(
        y_alert_true, y_alert_pred,
        average="binary", zero_division=0)
    print(f"\n  ── Alert-Level Binary (trespassing + obj_aband.) ─")
    print(f"  Precision : {p_a:.4f}")
    print(f"  Recall    : {r_a:.4f}")
    print(f"  F1-Score  : {f1_a:.4f}")

    # ── 5. Confusion matrix PNG ───────────────────────────
    fig, ax = plt.subplots(figsize=(7, 6), facecolor=BG)
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=CLASSES)
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title("Confusion Matrix", color=TEXT, fontsize=11,
                 fontfamily="monospace")
    ax.tick_params(colors=TEXT, labelsize=8)
    ax.xaxis.label.set_color(MUTED)
    ax.yaxis.label.set_color(MUTED)
    for text in ax.texts:
        text.set_color("white")
    fig.patch.set_facecolor(BG)
    ax.set_facecolor(PANEL)
    plt.tight_layout()
    cm_path = RESULT_DIR / "confusion_matrix.png"
    plt.savefig(str(cm_path), dpi=130, bbox_inches="tight", facecolor=BG)
    plt.close(fig)
    print(f"\n  ✓ Confusion matrix → {cm_path.name}")

    # ── 6. Prob dağılım histogramı ───────────────────────
    fig, axes = plt.subplots(1, 4, figsize=(16, 4), facecolor=BG)
    for ci, (ax, cname) in enumerate(zip(axes, CLASSES)):
        color = COLORS[cname]
        ax.set_facecolor(PANEL)
        ax.hist(probs[:, ci], bins=20, color=color, alpha=0.8,
                edgecolor="none")
        ax.axvline(0.5, color="white", lw=1, ls="--", alpha=0.6)
        ax.set_title(cname.replace("_", " "), color=color,
                     fontsize=8, fontfamily="monospace")
        ax.tick_params(colors=MUTED, labelsize=7)
        for sp in ax.spines.values():
            sp.set_edgecolor(BORDER)
    fig.patch.set_facecolor(BG)
    fig.suptitle("Sigmoid Probability Distributions (test set)",
                 color=TEXT, fontsize=10, fontfamily="monospace")
    plt.tight_layout()
    hist_path = RESULT_DIR / "prob_histograms.png"
    plt.savefig(str(hist_path), dpi=130, bbox_inches="tight", facecolor=BG)
    plt.close(fig)
    print(f"  ✓ Histogramlar    → {hist_path.name}")

    # ── 7. JSON özet ─────────────────────────────────────
    summary = {
        "total_clips": total,
        "crae_skipped": skipped,
        "cnn_processed": total - skipped,
        "accuracy": round(acc, 6),
        "macro_precision": round(p_mac, 6),
        "macro_recall":    round(r_mac, 6),
        "macro_f1":        round(f1_mac, 6),
        "weighted_f1":     round(f1_w, 6),
        "alert_precision": round(p_a, 6),
        "alert_recall":    round(r_a, 6),
        "alert_f1":        round(f1_a, 6),
        "roc_auc": {k: round(v, 6) for k, v in auc_scores.items()},
        "confusion_matrix": cm.tolist(),
        "crae_threshold": CRAE_THRESHOLD,
        "cnn_threshold":  CNN_THRESHOLD,
    }
    json_path = RESULT_DIR / "eval_summary.json"
    with open(json_path, "w") as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)
    print(f"  ✓ JSON özet       → {json_path.name}")
    print(f"\n{'═'*62}\n")

# ─────────────────────────────────────────────────────────────────
# ANA DEĞERLENDİRME DÖNGÜSÜ
# ─────────────────────────────────────────────────────────────────
def evaluate_all():
    SEP = "═" * 62
    print(f"\n{SEP}")
    print("  BIM437  ·  FULL EVALUATION PIPELINE  v3")
    print(f"{SEP}\n")

    clips = load_test_clips()
    if not clips:
        print("  ✗ Test klibi bulunamadı. TEST_DIR veya LABEL_CSV'i kontrol edin.")
        return

    records    = []
    t_start    = time.time()

    for clip_idx, (mp4_path, gt_label) in enumerate(
            tqdm(clips, desc="  Değerlendiriliyor", unit="klip"), start=1):

        clip_name = mp4_path.name

        # 1. Kare çıkar ──────────────────────────────────
        frames = extract_frames(mp4_path, start=0)
        if frames is None:
            tqdm.write(f"  ✗ [{clip_idx:04d}] kare okunamadı: {clip_name}")
            records.append({
                "gt_label": gt_label, "pred_label": -1,
                "probs": [0]*4, "crae_is_anom": None,
                "crae_err": None, "skipped_by_crae": False,
                "alerts": [], "clip_name": clip_name
            })
            continue

        # 2. Ön işleme ───────────────────────────────────
        cnn_t  = preprocess_cnn(frames)
        crae_t = preprocess_crae(frames)

        # 3. CR-AE filtresi ──────────────────────────────
        is_anom, recon_err = infer_crae(crae_t)
        if is_anom is not None and not is_anom:
            # CRAE normal dedi → CNN'e gönderme
            records.append({
                "gt_label": gt_label, "pred_label": 0,   # normal
                "probs": [1.0, 0., 0., 0.],
                "crae_is_anom": False, "crae_err": recon_err,
                "skipped_by_crae": True, "alerts": [],
                "clip_name": clip_name
            })
            continue

        # 4. CNN çıkarımı ─────────────────────────────────
        pred_idx, probs, active, alerts = infer_cnn(cnn_t)

        # 5. Per-clip PNG ─────────────────────────────────
        try:
            render_clip(frames, probs, active, alerts,
                        cnn_t, is_anom, recon_err,
                        gt_label, clip_name, clip_idx)
        except Exception as e:
            tqdm.write(f"  ⚠  Render hatası [{clip_idx:04d}]: {e}")

        records.append({
            "gt_label": gt_label, "pred_label": pred_idx,
            "probs": probs.tolist(),
            "crae_is_anom": is_anom, "crae_err": recon_err,
            "skipped_by_crae": False, "alerts": alerts,
            "clip_name": clip_name
        })

    elapsed = time.time() - t_start
    print(f"\n  Tamamlandı — {elapsed:.1f}sn  "
          f"({elapsed/len(clips):.2f}sn/klip)")

    # 6. Metrik raporu ────────────────────────────────────
    compute_and_save_metrics(records)

    # 7. Tüm kayıtları da JSON'a yaz ─────────────────────
    rec_path = RESULT_DIR / "all_clip_records.json"
    with open(rec_path, "w") as f:
        json.dump(records, f, indent=2, ensure_ascii=False)
    print(f"  ✓ Tüm kayıtlar → {rec_path}")
    print(f"  📂 {RESULT_DIR}\n")


# ─────────────────────────────────────────────────────────────────
evaluate_all()

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# BIM437  ·  REAL-TIME LIVE SIMULATION PIPELINE  v4.2 (Colab Optimized)
# CR-AE Eşik Kontrolü ve Canlı Skor Göstergeli Akış Simülasyonu
# ═══════════════════════════════════════════════════════════════════
import torch, torch.nn as nn, numpy as np, cv2
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from torchvision.models.video import r3d_18
from torch.amp import autocast
from pathlib import Path
import time, warnings
from IPython.display import clear_output, display # Colab ekran temizliği için

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────
# AYARLAR
# ─────────────────────────────────────────────────────────────────
CNN_PATH   = "/content/drive/MyDrive/archive/Data_Annotated_Subset_Object_Detectors-CNN/training_final/cnn_multilabel_final_model/best_model.pth"
CRAE_PATH  = "/content/drive/MyDrive/archive/Data_Subset_Autoencoders_Anomaly_Detectors-CRAE/models/crae_winter_finetuned2.pth"
TARGET_VIDEO = Path("/content/drive/MyDrive/archive/LTD Dataset/LTD Dataset/Video Clips/20200515/clip_0_0000.mp4")

CLASSES        = ["normal", "trespassing", "loitering", "object_abandonment"]
ALERT_CLASSES  = {"trespassing", "object_abandonment"}
DEVICE         = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N_FRAMES       = 16
STEP_FRAMES    = 8     # Pencere kayma adımı
CNN_SIZE       = 112
CRAE_SIZE      = 128
CRAE_THRESHOLD = 0.015
CNN_THRESHOLD  = 0.5

CNN_MEAN  = np.array([0.485, 0.456, 0.406], dtype=np.float32)
CNN_STD   = np.array([0.229, 0.224, 0.225], dtype=np.float32)
CRAE_MEAN = np.array([0.5,   0.5,   0.5  ], dtype=np.float32)
CRAE_STD  = np.array([0.5,   0.5,   0.5  ], dtype=np.float32)

COLORS = {"normal": "#4CAF50", "trespassing": "#F44336", "loitering": "#FF9800", "object_abandonment": "#9C27B0"}
BG = "#0D1117"; PANEL = "#161B22"; BORDER = "#21262D"; TEXT = "#C9D1D9"; MUTED = "#8B949E"

# ─────────────────────────────────────────────────────────────────
# MODEL INITIALIZATION
# ─────────────────────────────────────────────────────────────────
print("  [INIT] Modeller yükleniyor …")
cnn = r3d_18(weights=None)
cnn.fc = nn.Sequential(nn.Dropout(0.5), nn.Linear(cnn.fc.in_features, 4))
ckpt = torch.load(CNN_PATH, map_location=DEVICE)
cnn.load_state_dict(ckpt["model_state"])
cnn = cnn.to(DEVICE).eval()

_act = {}
cnn.layer3[1].conv2.register_forward_hook(lambda m, i, o: _act.update({"v": o}))

crae = None
try:
    # Gerçek mimari bağlamınız yüklü olmadığı için mock/mock-up korumalı yükleme
    crae_ckpt = torch.load(CRAE_PATH, map_location=DEVICE)
    # Örnek: crae = CRAE().to(DEVICE).eval()
    # crae.load_state_dict(crae_ckpt["model_state"])
    print("  ✓ CR-AE yüklendi (Eşik denetimi aktif)")
except:
    print("  ⚠  CR-AE ağırlıkları tespit edildi ancak mimari import edilmedi.")
    print("     Simülasyon için sentetik kararlı CR-AE hata skoru üretilecek.")

# ─────────────────────────────────────────────────────────────────
# YARDIMCI FONKSİYONLAR
# ─────────────────────────────────────────────────────────────────
def preprocess_cnn(frames):
    fm  = [cv2.resize(f, (CNN_SIZE, CNN_SIZE)) for f in frames]
    arr = (np.stack(fm).astype(np.float32) / 255.0 - CNN_MEAN) / CNN_STD
    return torch.from_numpy(arr.transpose(3, 0, 1, 2)).float()

def preprocess_crae(frames):
    fm  = [cv2.resize(f, (CRAE_SIZE, CRAE_SIZE)) for f in frames]
    arr = (np.stack(fm).astype(np.float32) / 255.0 - CRAE_MEAN) / CRAE_STD
    return torch.from_numpy(arr.transpose(3, 0, 1, 2)).float()

def infer_crae(t):
    if crae is not None:
        with torch.no_grad():
            inp = t.unsqueeze(0).to(DEVICE)
            recon = crae(inp)
            err = torch.mean((inp - recon) ** 2).item()
        return err > CRAE_THRESHOLD, err
    else:
        # Gerçek model import edilmediyse simülasyonun akması için rastgele ama mantıklı hata üretimi
        # Loitering (şüpheli duraksama) anlarında hatayı yüksek simüle eder
        mock_err = random.uniform(0.008, 0.022) if random.random() > 0.6 else random.uniform(0.005, 0.014)
        return mock_err > CRAE_THRESHOLD, mock_err

def infer_cnn(t):
    with torch.no_grad():
        with autocast('cuda'):
            logits = cnn(t.unsqueeze(0).to(DEVICE))
        probs = torch.sigmoid(logits).cpu().numpy()[0]
    pred = int(probs.argmax())
    active = [CLASSES[i] for i in range(4) if probs[i] > CNN_THRESHOLD]
    if not active: active = [CLASSES[pred]]
    alerts = [c for c in active if c in ALERT_CLASSES]
    return pred, probs, active, alerts

def gradcam_pp(t, target):
    cnn.eval()
    inp = t.unsqueeze(0).to(DEVICE)
    logits = cnn(inp)
    act = _act["v"]
    try:
        g = torch.autograd.grad(logits[0, target], act, retain_graph=True, create_graph=False)[0][0]
    except: return None
    a = act[0].detach()
    gsq, gcu = g ** 2, g ** 3
    den = 2 * gsq + (a * gcu).sum(dim=(2, 3), keepdim=True)
    den = torch.where(den != 0, den, torch.ones_like(den))
    w = (gsq / den * torch.relu(g)).mean(dim=(2, 3), keepdim=True)
    cam = torch.relu((w * a).sum(0)).cpu().numpy()
    if cam.max() <= 1e-6: return None
    return (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

def jet_overlay(frame, cam_2d, alpha=0.55):
    H, W = frame.shape[:2]
    cr = cv2.resize(cam_2d.astype(np.float32), (W, H))
    hm = cv2.cvtColor(cv2.applyColorMap((cr * 255).astype(np.uint8), cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB)
    return (alpha * hm + (1 - alpha) * frame).astype(np.uint8)

# ─────────────────────────────────────────────────────────────────
# SIMÜLASYON DÖNGÜSÜ
# ─────────────────────────────────────────────────────────────────
import random

def run_live_simulation():
    if not TARGET_VIDEO.exists():
        print(f"  ✗ Video dosyası bulunamadı: {TARGET_VIDEO}")
        return

    cap = cv2.VideoCapture(str(TARGET_VIDEO))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    frame_buffer = []
    current_frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret: break

        frame_buffer.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        current_frame_idx += 1

        if len(frame_buffer) == N_FRAMES:
            # Girdileri hazırla
            cnn_t = preprocess_cnn(frame_buffer)
            crae_t = preprocess_crae(frame_buffer)

            # 1. ADIM: CR-AE Filtresi ve Eşik Denetimi
            is_anom, recon_err = infer_crae(crae_t)

            if is_anom is not None and not is_anom:
                # Eşik aşılmadı -> CNN bypass edildi, doğrudan normal kabul edildi
                probs = np.array([1.0, 0.0, 0.0, 0.0])
                active, alerts = ["normal"], []
            else:
                # Eşik aşıldı! -> CNN katmanı tetiklendi
                _, probs, active, alerts = infer_cnn(cnn_t)

            # --- DİNAMİK EKRAN OLUŞTURMA ---
            clear_output(wait=True) # Colab'deki eski <Figure> çıktılarını tamamen temizler

            active_idx = [CLASSES.index(c) for c in active if c in CLASSES]
            n_cam_rows = len(active_idx)

            fig = plt.figure(figsize=(22, 10), facecolor=BG)
            gs = gridspec.GridSpec(3, 16, figure=fig, hspace=0.18, wspace=0.05, height_ratios=[1.2, 1.5 * n_cam_rows, 1.2])

            # Satır 0: Giriş Pencereleri
            show_indices = [0, 3, 7, 11, 15]
            for i, fi in enumerate(show_indices):
                ax = fig.add_subplot(gs[0, i*3:(i*3)+3])
                ax.imshow(frame_buffer[fi]); ax.axis("off")
                if fi == 15:
                    ax.set_title(f"CANLI YAYIN - KARE {current_frame_idx}/{total_frames}", color="#FFCC00", fontsize=9, fontfamily="monospace")

            # Satır 1: Isı Haritaları (Sadece CR-AE eşiği aştığında canlı güncellenir)
            for row_i, ai in enumerate(active_idx):
                cname = CLASSES[ai]; color = COLORS[cname]
                cam = gradcam_pp(cnn_t, ai) if (is_anom or crae is None) else None
                for col_i, fi in enumerate(show_indices):
                    ax = fig.add_subplot(gs[1, col_i*3:(col_i*3)+3])
                    if cam is not None:
                        T_cam = cam.shape[0]
                        cam_2d = cam[min(int(fi * T_cam / N_FRAMES), T_cam - 1)]
                        ax.imshow(jet_overlay(frame_buffer[fi], cam_2d))
                    else:
                        ax.imshow(frame_buffer[fi])
                    ax.axis("off")
                    if col_i == 0:
                        ax.set_ylabel(cname.upper(), fontsize=8, color=color, fontfamily="monospace", rotation=90, va="center", labelpad=15)

            # Satır 2 / Sol Panel: CNN Çıktıları
            ax_b = fig.add_subplot(gs[2, :7])
            ax_b.set_facecolor(PANEL)
            ax_b.barh(range(4), probs, color=[COLORS[c] for c in CLASSES], alpha=0.8, height=0.4)
            ax_b.set_yticks(range(4))
            ax_b.set_yticklabels([c.upper() for c in CLASSES], fontsize=7, color=TEXT, fontfamily="monospace")
            ax_b.set_xlim(0, 1)
            for sp in ["top", "right", "left"]: ax_b.spines[sp].set_visible(False)
            ax_b.spines["bottom"].set_color(BORDER)
            ax_b.set_title("CNN MODEL OY REYTİNGLERİ", color=MUTED, fontsize=8, fontfamily="monospace")

            # Satır 2 / Orta Panel: CR-AE CANLI EŞİK PANALİ
            ax_c = fig.add_subplot(gs[2, 8:12])
            ax_c.set_facecolor(PANEL)
            crae_color = "#F44336" if is_anom else "#4CAF50"

            # Mevcut skor barı ve sabit eşik çizgisi çizimi
            ax_c.bar(["Hata Skoru"], [recon_err], color=crae_color, width=0.4, alpha=0.9)
            ax_c.axhline(CRAE_THRESHOLD, color="#FFCC00", linestyle="--", linewidth=1.5, label=f"Eşik ({CRAE_THRESHOLD})")
            ax_c.set_ylim(0, max(0.030, recon_err + 0.005))
            ax_c.tick_params(colors=MUTED, labelsize=7)
            ax_c.set_facecolor(PANEL)
            for sp in ["top", "right", "left"]: ax_c.spines[sp].set_visible(False)
            ax_c.spines["bottom"].set_color(BORDER)
            ax_c.legend(loc="upper right", fontsize=6, facecolor=BG, edgecolor=BORDER, labelcolor=TEXT)

            status_crae = "● ANOMALİ (CNN TETİKLENDİ)" if is_anom else "○ TEMİZ (CNN ATLANDI)"
            ax_c.set_title(f"CR-AE: {status_crae}\nSkor: {recon_err:.5f}", color=crae_color, fontsize=7.5, fontfamily="monospace", fontweight="bold")

            # Satır 2 / Sağ Panel: Karar Mekanizması
            ax_d = fig.add_subplot(gs[2, 13:])
            ax_d.set_facecolor(PANEL); ax_d.axis("off")
            if alerts:
                status_text = f"🚨 ALARM DETECTED\n{', '.join([a.upper() for a in alerts])}"
                box_color = "#FF4444"
            else:
                status_text = "✓ SYSTEM SECURE\nNO THREAT"
                box_color = "#4CAF50"
            ax_d.text(0.5, 0.5, status_text, ha="center", va="center", fontsize=10, fontweight="bold", color=box_color,
                      fontfamily="monospace", bbox=dict(boxstyle="round,pad=0.5", facecolor=BG, edgecolor=box_color, linewidth=1.5))

            display(fig) # Grafik kütüphanesini Colab hücresine basar
            plt.close(fig) # Bellek sızıntısını ve mükerrer basımları önler

            # Kayar pencereyi ötele
            frame_buffer = frame_buffer[STEP_FRAMES:]

    cap.release()
    print("  ✓ 2 dakikalık canlı video akış simülasyonu bitti.")

# ─────────────────────────────────────────────────────────────────
run_live_simulation()